In [136]:
import pickle
import numpy as np 
import re 
from pathlib import Path
import pandas as pd
import json 
%matplotlib inline 
import matplotlib
import matplotlib.pyplot as plt 
import seaborn as sns
from src import util_analysis 

# from matplotlib.ticker import FormatStrFormatter
import re 

In [137]:
matplotlib.rcParams.update({'font.size': 10})
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'

fig_out_dir = Path("final_figures/figure_2")
fig_out_dir.mkdir(exist_ok=True, parents=True)

## Plot results from Popham-style conditions using SWC stimuli

### Import human data

In [138]:
path_to_human_data = Path('/mindhive/mcdermott/www/imgriff/msjspsych/')
# !ls {path_to_human_data}
# full paths to .json files 
human_fnames = list(path_to_human_data.glob("cocktail_party_popham_swc_word_recognition/data/*.json"))

# import vocab dict for matching audio & responses 
# word_and_speaker_encodings = pickle.load( open("/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
# # ix_to_word_map = {v:k for k,v in class_map.items()}
# class_map = word_and_speaker_encodings['word_idx_to_word']
len(human_fnames)


236

## Process Human Results

In [139]:
## Get all subject data into one df for analysis

def get_part_df(fname):
    part_data = json.load(open(fname, 'r'))
    # print(f"{fname.stem} success {part_data[0]['success']}")
    part_df = pd.DataFrame.from_records(part_data)
    ## Forward fill stim presentation entry to word response entry
    responses = part_df.loc[part_df.trial_type.isin(['audio-keyboard-response','dictionary-text']), ['trial_index', 'stimulus']]
    responses = responses.ffill()
    part_df.loc[part_df['trial_index'].isin(responses["trial_index"].values), 'stimulus'] = responses.stimulus
    return part_df

dfs = []
for fname in human_fnames:
    try:
        dfs.append(get_part_df(fname))
    except Exception as e:
        # print(e)
        # print(fname)
        continue
    
# results_df = pd.concat([get_part_df(fname) for fname in human_fnames], axis=0, ignore_index=True)
results_df = pd.concat(dfs)
print(f"Total number of subjects: {len(results_df.id_subject.unique())}")


## Filter for those who passed headphone check 
results_df = results_df[results_df.hc_passed == True]
print(f"Total number of subjects after headphone check: {len(results_df.id_subject.unique())}")


## Add snr and condition names as columns - unpack from file code in file names 

import re
### Get condition map
with open('swc_popham_exmpt_2024_cond_manifest.pkl', 'rb') as f:
    stim_cond_map = pickle.load(f)

stim_cond_map = {f"condition_{k:02}": v for k,v in stim_cond_map.items()}

## Map wav str to stim_type using condition dict
def get_stim_snr_and_cond(stim_str, stim_cond_map=stim_cond_map):
    target_harm, dist_harm = None,  None 
    if isinstance(stim_str, str) and not stim_str.startswith('<'):
        # print(stim_str)
        cond_str = re.search("condition_(-?\d+)", stim_str)
        if cond_str:
            cond_str = cond_str.group(0)
            target_harm = stim_cond_map[cond_str]['target_harmonicity']
            dist_harm = stim_cond_map[cond_str]['distractor_harmonicity']
            if dist_harm is None:
                dist_harm = 'No Distractor'
            target_harm = target_harm.title()
            dist_harm = dist_harm.title()
        elif 'catch' in stim_str:
            target_harm = 'catch_trial'
            dist_harm = 'No Distractor'
    return target_harm, dist_harm

# add as columns 
results_df['target_harmonicity'], results_df['distractor_harmonicity'] = zip(*results_df['stimulus'].apply(get_stim_snr_and_cond))

# cut down df to only have trial answers (remove other meta data)
expmnt_trial_str = "dictionary-text"
trial_results = results_df[results_df.trial_type == expmnt_trial_str]


# screen based on completion 
total_trials = 192 
full_run_subjects = [subj_id for subj_id, did_all_trials in (trial_results.groupby('id_subject').target_harmonicity.count() == total_trials).items() if did_all_trials == True]
trial_results = trial_results[trial_results.id_subject.isin(full_run_subjects)]

# # Add accuracy 
trial_results['accuracy'] = (trial_results['response'] == trial_results['correct_response']).astype('int')
print(f"Total number of subjects after full run check: {len(trial_results.id_subject.unique())}")

Total number of subjects: 221
Total number of subjects after headphone check: 153
Total number of subjects after full run check: 110


In [140]:
# trial_results.groupby('id_subject').target_harmonicity.count()

In [141]:
catch_trial_particiapnt_performance = trial_results[trial_results.target_harmonicity == 'catch_trial'].groupby('id_subject').accuracy.mean()
print("N total participants ", len(catch_trial_particiapnt_performance))
good_participants = catch_trial_particiapnt_performance[catch_trial_particiapnt_performance >= 11/12].index
print("N good participants ", len(good_participants))

good_results = trial_results[trial_results.id_subject.isin(good_participants)]

analysis_df = good_results[(~good_results.target_harmonicity.isnull()) & (~good_results.response.isna())]
analysis_df = analysis_df[analysis_df.target_harmonicity != 'catch_trial']

# analysis_df.groupby('id_subject').target_harmonicity.count()

N total participants  110
N good participants  90



## Add re-scored performance based on entries included in excerpt transcripts  

In [143]:
meta_df = pd.read_pickle('/om/user/imgriff/datasets/human_swc_popham_exmpt_2024/source_stim_meta_manifest.pdpkl')
meta_df['stim_name'] = meta_df.target_fn.apply(lambda x: Path(x).stem) + "_" + meta_df.distractor_fn.apply(lambda x: Path(x).stem) 
analysis_df = pd.merge(analysis_df,
                        meta_df[['word', 'word_int', 'distractor_word', 'target_transcripts', 'distractor_transcripts', 'gender_cond', 'stim_name']],
                        left_on=['correct_response'], right_on=['word'], how='left')     
                        
analysis_df['confusions'] = (analysis_df.response == analysis_df.distractor_word).astype('int')
# add adjusted accuracy and confusions 
target_words = analysis_df.response.values
target_transcripts = analysis_df.target_transcripts.values
distractor_transcripts = analysis_df.distractor_transcripts.values
distractor_words = analysis_df.distractor_word.values


adjusted_acc = np.array([int(target_word in target_transcript)
                            if not isinstance(target_transcript, float) else np.nan
                             for target_word, target_transcript in zip(target_words, target_transcripts)
                              ])

adjusted_confs = np.array([int((target_word in distractor_transcript) or (target_word == distractor_word))
                            if not isinstance(distractor_transcript, float) else np.nan
                             for target_word, distractor_word, distractor_transcript in zip(target_words, distractor_words, distractor_transcripts)
                              ])

analysis_df['adjusted_accuracy'] = adjusted_acc
analysis_df['adjusted_confusions'] = adjusted_confs
analysis_df['stim_name'] = analysis_df['stim_name'] + "_"+ analysis_df['target_harmonicity'].str.lower().str.replace(' ', '_') + "_" + analysis_df['distractor_harmonicity'].str.lower().str.replace(' ', '_') + "_" + analysis_df['gender_cond']

In [144]:
analysis_df['stim_name'].value_counts()

activities_vaslittlecrow_worked_enochlau_whispered_whispered_different_sex             11
radio_persian-poet-gal_other_popularoutcast_inharmonic_inharmonic_same_sex             11
character_theexistentialist686_record_batwoodman_inharmonic_whispered_different_sex    11
trade_patrickjoel_lines_thatgirltayler_harmonic_whispered_different_sex                10
place_montrealais_after_flyingtoaster_harmonic_harmonic_different_sex                  10
                                                                                       ..
public_s-whistler_united_jonjalin_inharmonic_harmonic_different_sex                     1
water_popularoutcast_another_ama1016_inharmonic_no_distractor_same_sex                  1
local_hassocks5489_because_s-whistler_whispered_harmonic_same_sex                       1
track_potteryfreak_political_flyingtoaster_whispered_whispered_same_sex                 1
related_dolliellama_after_persian-poet-gal_harmonic_harmonic_same_sex                   1
Name: stim

In [103]:
analysis_df.stim_name.value_counts().describe()

count    4219.000000
mean        3.839772
std         1.814650
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        11.000000
Name: stim_name, dtype: float64

In [104]:
# get trial data excluding catch trials 


part_summary_df = (analysis_df.groupby(['stim_name', "target_harmonicity", 'distractor_harmonicity'])
                     .agg({'correct':['mean', 'sem'],
                            'confusions':['mean', 'sem'],
                            'adjusted_accuracy':['mean', 'sem'],
                            'adjusted_confusions':['mean', 'sem', 'count']})
                     .reset_index())

# flatten multiindex 
part_summary_df.columns = ['_'.join(col).strip() for col in part_summary_df.columns.values]
# remove trailing underscore
part_summary_df.columns = [col[:-1] if col.endswith('_') else col for col in part_summary_df.columns.values]


In [108]:
part_summary_df.stim_name

0       above_flyingtoaster_were_madridteacher1_differ...
1       above_flyingtoaster_were_madridteacher1_differ...
2       above_flyingtoaster_were_madridteacher1_differ...
3       above_flyingtoaster_were_madridteacher1_differ...
4       above_flyingtoaster_were_madridteacher1_differ...
                              ...                        
4214    years_theroachyjay_state_persian-poet-gal_diff...
4215    years_theroachyjay_state_persian-poet-gal_diff...
4216    years_theroachyjay_state_persian-poet-gal_diff...
4217    years_theroachyjay_state_persian-poet-gal_diff...
4218    years_theroachyjay_state_persian-poet-gal_diff...
Name: stim_name, Length: 4219, dtype: object

## Set paths and load model results 

In [43]:
## import class maps
import pickle
## load WSN vocab mapping 
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
wsn_word_2_class = word_and_speaker_encodings['word_to_idx']
wsn_class_2_word = word_and_speaker_encodings['word_idx_to_word']
cv_word_2_class = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
cv_class_2_word = {v:k for k,v in cv_word_2_class.items()}


In [122]:
# parent_path = Path('/om2/user/imgriff/projects/torch_2_aud_attn/popham_mono_eval/')
parent_path = Path('/om2/user/imgriff/projects/torch_2_aud_attn/popham_swc_eval_all_stim/')

# model_name = 'word_task_standard_v08'
# model_name = 'word_task_half_co_loc_v07'

meta_df = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/human_cue_target_distractor_df_w_meta_transcripts.pdpkl')
meta_df['expt_df_ix'] = meta_df.index
cols_to_merge = [
                 'gender',
                 'word',
                 'target_transcripts',
                 'same_sex_distractor_1_transcripts',
                 'diff_sex_distractor_1_transcripts',
                 "same_sex_dist_1_word",
                 "diff_sex_dist_1_word",
                 "excerpt_src_fn",
                 "same_sex_distractor_1_src_fn",
                 "diff_sex_distractor_1_src_fn",
                 "expt_df_ix"
]

results_dirs = list(parent_path.rglob(f"*/*v10*.csv"))
# results_dirs += list(parent_path.rglob(f"*/*50Hz_cutoff*.csv"))
print(len(results_dirs))

dfs = []

for result_csv in results_dirs:
    if 'arch_' in str(result_csv):
        continue
    # print(result_csv)
    # if not 'word_task_half_co_loc_v09_gender_bal_4M_w_no_cue_learned_higher_lr_less_dropout' in str(result_csv) or not '_arch_' in str(result_csv):
    #     # print('skp')
    #     continue
    try:
        df = pd.read_csv(result_csv)
    except Exception as e:
        print(e)
        print(result_csv)
        continue
    
    # break 
    df['target_harmonicity'] = result_csv.stem.split('_target')[0].split('_')[-1].title()
    df['model'] = result_csv.parent.stem
    dist_harm = result_csv.stem.split('_distractor')[0].split('_')[-1].title()
    if dist_harm == 'None':
        dist_harm = 'No Distractor'
    df['distractor_harmonicity'] = dist_harm
    df['target_word'] = df['true_word_int'].replace(cv_class_2_word)
    df['pred_word'] = df['pred_word_int'].replace(cv_class_2_word)
    df = pd.merge(df,
                 meta_df[cols_to_merge],
                 left_on=['target_word','orig_df_row_ix'], right_on=['word', 'expt_df_ix'], how='left')     
    dfs.append(df)

model_results = pd.concat(dfs, axis=0, ignore_index=True)
model_results['confusions'] = (model_results.pred_word == model_results.same_sex_dist_1_word).astype('int') + (model_results.pred_word == model_results.diff_sex_dist_1_word).astype('int')
model_results['accuracy'] = (model_results.pred_word == model_results.target_word).astype('int')
# add adjusted accuracy and confusions 
pred_words = model_results.pred_word.values
target_words = model_results.word.values
target_transcripts = model_results.target_transcripts.values
same_sex_distractor_words = model_results.same_sex_dist_1_word.values
diff_sex_distractor_words = model_results.diff_sex_dist_1_word.values
same_sex_distractor_transcripts = model_results.same_sex_distractor_1_transcripts.values
diff_sex_distractor_transcripts = model_results.diff_sex_distractor_1_transcripts.values


adjusted_acc = np.array([int(pred_word in target_transcript or pred_word == target_word)
                            if not isinstance(target_transcript, float) else np.nan
                            for pred_word, target_word, target_transcript in zip(pred_words, target_words, target_transcripts)
                            ])

adjusted_confs = np.array([int(pred_word in same_sex_transcript or pred_word in diff_sex_transcript or pred_word == same_sex_word or pred_word == diff_sex_word)
                            if not (isinstance(same_sex_transcript, float) and isinstance(diff_sex_transcript, float)) else np.nan
                            for pred_word, same_sex_word, diff_sex_word, same_sex_transcript, diff_sex_transcript in zip(pred_words, same_sex_distractor_words, diff_sex_distractor_words,  same_sex_distractor_transcripts, diff_sex_distractor_transcripts)
                            ])

model_results['adjusted_accuracy'] = adjusted_acc
model_results['adjusted_confusions'] = adjusted_confs


162


In [175]:
## Get model stim_name 
def manage_dist_str(dist_str):
    dist_str = dist_str.replace("_eg_0", '').split('_')[-1].replace('-', '_', 1) 
    return dist_str

def get_stim_name(df_row):
    target_str = Path(df_row.excerpt_src_fn).stem.split('_')[-1].replace('-', '_', 1) 
    if df_row.sex_cond_int == 0: # is same sex
        dist_str = manage_dist_str(Path(df_row.same_sex_distractor_1_src_fn).stem)
        sex_cond = 'same_sex'
    else:  
        dist_str = manage_dist_str(Path(df_row.diff_sex_distractor_1_src_fn).stem)
        sex_cond = 'different_sex'
    target_harmonicity = df_row.target_harmonicity.lower().replace(' ', '_')
    distractor_harmonicity = df_row.distractor_harmonicity.lower().replace(' ', '_')
    stim_name = f"{target_str}_{dist_str}_{target_harmonicity}_{distractor_harmonicity}_{sex_cond}"
    return stim_name
model_results['stim_name'] = model_results.apply(get_stim_name, axis=1)


In [177]:
model_results.stim_name.value_counts().describe()

count    23424.000000
mean         4.500000
std          0.500011
min          4.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: stim_name, dtype: float64

In [178]:
analysis_df.stim_name

0        research_mangst_after_gorillawarfare_whispered...
1        association_etoile_school_sethallen623_inharmo...
2        already_popularoutcast_without_luigi30_whisper...
3        parts_alexkillby_politics_lily5lace_whispered_...
4        commercial_warmvoiceover_species_macropode_inh...
                               ...                        
16195    already_popularoutcast_without_luigi30_inharmo...
16196    taken_flyingtoaster_divided_popularoutcast_whi...
16197    feature_flyingtoaster_leading_jasonlamarche_ha...
16198    rights_dolliellama_birds_laura-domineau_inharm...
16199    original_scott-burley_college_popularoutcast_i...
Name: stim_name, Length: 16200, dtype: object

In [179]:
model_results.stim_name

0         human_wodup_strong_divinemeaninglessness_harmo...
1         human_wodup_flight_popularoutcast_harmonic_no_...
2         human_flyingtoaster_above_popularoutcast_harmo...
3         human_flyingtoaster_german_s-whistler_harmonic...
4         hundred_s-whistler_needed_thecyberwasp_harmoni...
                                ...                        
105403    taken_mangst_video_sophus-bie_whispered_whispe...
105404    taking_hassocks5489_natural_alexkillby_whisper...
105405    taking_hassocks5489_number_popularoutcast_whis...
105406    taking_flyingtoaster_german_potteryfreak_whisp...
105407    taking_flyingtoaster_could_the-voice-of-hassoc...
Name: stim_name, Length: 105408, dtype: object

In [180]:
# --- Check overlap between human and model stim --- 
human_stims = set(analysis_df.stim_name.unique())
model_stims = set(model_results.stim_name.unique())

intersection = model_stims.intersection(human_stims)
intersection

set()

In [119]:
model_meta_df = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/human_cue_target_distractor_df_w_meta_transcripts.pdpkl')
model_meta_df.excerpt_src_fn.apply(lambda x: Path(x).stem.split('_')[-1].replace('-', '_', 1)) \
 + "_" \
 + model_meta_df.distractor_fn.apply(lambda x: Path(x).stem.split('_')[-1].replace('-', '_', 1)) \
 

AttributeError: 'DataFrame' object has no attribute 'distractor_fn'

### Conform data to plot 

In [11]:
### Summarize models like participants 

model_summary_df = (model_results.groupby(['model', "target_harmonicity", 'distractor_harmonicity']).agg({'accuracy':['mean', 'sem'],
                            'confusions':['mean', 'sem'],
                            'adjusted_accuracy':['mean', 'sem'],
                            'adjusted_confusions':['mean', 'sem', 'count']})
                     .reset_index())

# flatten multiindex 
model_summary_df.columns = ['_'.join(col).strip() for col in model_summary_df.columns.values]
# remove trailing underscore
model_summary_df.columns = [col[:-1] if col.endswith('_') else col for col in model_summary_df.columns.values]

In [12]:

melted_model_results = pd.melt(model_summary_df, id_vars=['target_harmonicity', 'model', 'distractor_harmonicity'],
        value_vars=['adjusted_accuracy_mean', 'adjusted_confusions_mean'],
        value_name = 'hits',
        var_name = 'attended_stream')

# group_name_df = {
#                 'word_task_half_co_loc_v07': "Feature-based\nattention model",
#                  "word_task_v08_control_no_attn": "Baseline CNN",
#                  "word_task_half_co_loc_v09_50Hz_cutoff": "50Hz cutoff model",
#                  "word_task_half_co_loc_v08_gender_bal_4M_w_no_cue_learned_higher_lr_less_dropout": "v08 model",
#                 "word_task_conventional_layer_order": "Conventional Layer Order",

#                  "word_task_half_co_loc_v09_gender_bal_4M_w_no_cue_learned_higher_lr_less_dropout": "Feature-gain model"}

# group_name_df = util_analysis.model_name_dict


# melted_model_results['attended_stream'][melted_model_results['attended_stream'] == 'accuracy'] = "Cued stream"
# melted_model_results['attended_stream'][melted_model_results['attended_stream'] == 'confusions']= "Uncued stream"
melted_model_results.loc[melted_model_results['attended_stream'] == 'adjusted_accuracy_mean', 'attended_stream'] = "Target"
melted_model_results.loc[melted_model_results['attended_stream'] == 'adjusted_confusions_mean', 'attended_stream'] =  "Distractor"

IDX_no_dist_confs = melted_model_results.loc[(melted_model_results['distractor_harmonicity'] == 'No Distractor') & (melted_model_results.attended_stream == 'Distractor') ].index.values
melted_model_results = melted_model_results[~melted_model_results.index.isin(IDX_no_dist_confs)]
melted_model_results.loc[melted_model_results['distractor_harmonicity'] == 'No Distractor', 'attended_stream'] = "Single sentence"
melted_model_results.loc[melted_model_results.model.str.contains("early_only"), 'group'] = "Early-only model"
melted_model_results.loc[melted_model_results.model.str.contains("late_only"), 'group'] = "Late-only model"
melted_model_results.loc[melted_model_results.model.str.contains("control"), 'group'] = "Baseline CNN"
melted_model_results.loc[melted_model_results.model.str.contains("50Hz"), 'group'] = "50Hz model"
melted_model_results.loc[melted_model_results.model.str.contains("backbone"), 'group'] = "Computed-gain model"
melted_model_results.loc[~melted_model_results.model.str.contains("late|early|control|50Hz|backbone"), 'group'] = "Feature-gain model"

In [14]:
melted_participant_results = pd.melt(part_summary_df, id_vars=['id_subject', 'target_harmonicity', 'distractor_harmonicity'],
        value_vars=['adjusted_accuracy_mean', 'adjusted_confusions_mean'],
        value_name = 'hits',
        var_name = 'attended_stream')

melted_participant_results.loc[melted_participant_results['attended_stream'] == 'adjusted_accuracy_mean', 'attended_stream'] = "Target"
melted_participant_results.loc[melted_participant_results['attended_stream'] == 'adjusted_confusions_mean', 'attended_stream'] =  "Distractor"

IDX_no_dist_confs = melted_participant_results.loc[(melted_participant_results['distractor_harmonicity'] == 'No Distractor') & (melted_participant_results.attended_stream == 'Distractor') ].index.values
melted_participant_results = melted_participant_results[~melted_participant_results.index.isin(IDX_no_dist_confs)]
melted_participant_results.loc[melted_participant_results['distractor_harmonicity'] == 'No Distractor', 'attended_stream'] = "Single sentence"
melted_participant_results['group'] = f'Humans (N = {melted_participant_results.id_subject.nunique()})'